# Terminations in Microsoft Autogen

## Why Termination Matters

We have used team to do some work, say write a story or accomplish some task. 

In [4]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "gpt-4o",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)

In [6]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)

from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third]
)

In [ ]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
2

3
---------- TextMessage (add_1_agent_first) ----------
3
4
---------- TextMessage (add_1_agent_second) ----------
5
---------- TextMessage (add_1_agent_third) ----------
6


In [7]:
from autogen_agentchat.conditions import MaxMessageTermination

max_termination = MaxMessageTermination(5)

In [8]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    termination_condition =max_termination
)

In [5]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- add_1_agent_first ----------
1
---------- add_1_agent_second ----------
2
---------- add_1_agent_third ----------
3
---------- add_1_agent_first ----------
4
---------- add_1_agent_second ----------
5


TaskResult(messages=[TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=2), metadata={}, content='1', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=2), metadata={}, content='2', type='TextMessage'), TextMessage(source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=2), metadata={}, content='3', type='TextMessage'), TextMessage(source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=55, completion_tokens=2), metadata={}, content='4', type='TextMessage'), TextMessage(source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=60, completion_tokens=2), metadata={}, content='5', type='TextMessage')], stop_reason='Maximum number of messages 5 reached, current message count: 5')

In [9]:
from autogen_agentchat.agents import AssistantAgent

In [10]:

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

In [11]:
from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=text_mention_termination
)

In [12]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
Sir Ardent rode through storm‑torn valleys, sword gleaming. He faced the dragon, heart unwavering, slaying the beast to free the kingdom. Villagers sang his name, forever honoring his courage.
---------- TextMessage (story_critic) ----------
After peace, Ardent returned to the castle, teaching young knights the true meaning of bravery, ensuring the kingdom's legacy.  
Narrative strong, but dialogue could enrich character depth. Pacing slightly rushed; add more sensory detail.
---------- TextMessage (story_writer) ----------
Sir Ardent rode through storm‑torn valleys, sword flashing. He faced the dragon, heart unwavering, slaying the beast to free the kingdom. Villagers sang his name, forever honoring his courage.
---------- TextMessage (story_critic) ----------
Later, Ardent returned to the castle, guiding new knights in honor of the dragon’s memory. The tale is

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a story about a brave knight.', type='TextMessage'), TextMessage(source='story_writer', models_usage=RequestUsage(prompt_tokens=116, completion_tokens=545), metadata={}, content='Sir Ardent rode through storm‑torn valleys, sword gleaming. He faced the dragon, heart unwavering, slaying the beast to free the kingdom. Villagers sang his name, forever honoring his courage.', type='TextMessage'), TextMessage(source='story_critic', models_usage=RequestUsage(prompt_tokens=170, completion_tokens=426), metadata={}, content="After peace, Ardent returned to the castle, teaching young knights the true meaning of bravery, ensuring the kingdom's legacy.  \nNarrative strong, but dialogue could enrich character depth. Pacing slightly rushed; add more sensory detail.", type='TextMessage'), TextMessage(source='story_writer', models_usage=RequestUsage(prompt_tokens=220, completion_tokens=402), metadata={}, cont

# Combining Termination Conditions

In [13]:
combined_termination = MaxMessageTermination(5) | TextMentionTermination('THE END')

from autogen_agentchat.conditions import TextMentionTermination


text_mention_termination = TextMentionTermination('THE END')
teamWithTextTermination = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition=combined_termination
)

In [14]:
from autogen_agentchat.ui import Console

await Console(teamWithTextTermination.run_stream(task = 'Write a story about a brave knight.'))

---------- TextMessage (user) ----------
Write a story about a brave knight.
---------- TextMessage (story_writer) ----------
Sir Ardent rode into the storm‑shadowed valley, sword gleaming. He faced the dragon, heart unwavering, slaying it to free the kingdom. The people celebrated, forever honoring the knight whose courage forged peace.
---------- TextMessage (story_critic) ----------
THE END


TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a story about a brave knight.', type='TextMessage'), TextMessage(source='story_writer', models_usage=RequestUsage(prompt_tokens=649, completion_tokens=513), metadata={}, content='Sir Ardent rode into the storm‑shadowed valley, sword gleaming. He faced the dragon, heart unwavering, slaying it to free the kingdom. The people celebrated, forever honoring the knight whose courage forged peace.', type='TextMessage'), TextMessage(source='story_critic', models_usage=RequestUsage(prompt_tokens=709, completion_tokens=581), metadata={}, content='THE END', type='TextMessage')], stop_reason="Text 'THE END' mentioned")

# External Termination

ExternalTermination: Enables programmatic control of termination from outside the run. This is useful for UI integration (e.g., “Stop” buttons in chat interfaces).

In [16]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "family": "gpt-4o",
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
    },
)

In [17]:
from autogen_agentchat.agents import AssistantAgent

agent1 = AssistantAgent(
    name = 'story_writer',
    model_client=model_client,
    system_message="Give the story about a brave knight, keep it short no more than 40 words. if critic say 'THE END' anywhere. Only output 'THE END'"
)

agent2 = AssistantAgent(
    name = 'story_critic',
    model_client=model_client,
    system_message="Continue the story and critic it with feedback. Keep it short and no more than 40 words. If it feels complete, just say 'THE END'. Only output 'THE END'"
)

from autogen_agentchat.conditions import ExternalTermination
external_termination = ExternalTermination()

from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat(
    [agent1, agent2],
    termination_condition= external_termination
)



In [18]:
from autogen_agentchat.ui import Console
run = asyncio.create_task(Console(team.run_stream(task = 'Write a story about a brave knight less than 40 words.')))

await asyncio.sleep(10)

external_termination.set()
await run

---------- TextMessage (user) ----------
Write a story about a brave knight less than 40 words.
---------- TextMessage (story_writer) ----------
Sir Aric rode through misty valleys, visor gleaming. A dragon loomed, scales shimmering. With steadfast heart, he raised his sword and struck true. Villagers cheered; the knight returned, honor intact.
---------- TextMessage (story_critic) ----------
He journeyed back, heart light, knowing valor thrives beyond glory. The tale of Sir Aric spread, inspiring countless souls. **Feedback**: concise, vivid imagery; consider deeper motivation or twist for richer depth.
---------- TextMessage (story_writer) ----------
Sir Aric, driven by a lost promise, challenged the dragon atop the mountain. He struck true, freeing the village and redeeming his family's honor. Villagers cheered; his name echoed forever.
---------- TextMessage (story_critic) ----------
Years later, he returned, the legend alive in children's tales. The story ends strong, yet deeper f

TaskResult(messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a story about a brave knight less than 40 words.', type='TextMessage'), TextMessage(source='story_writer', models_usage=RequestUsage(prompt_tokens=121, completion_tokens=465), metadata={}, content='Sir Aric rode through misty valleys, visor gleaming. A dragon loomed, scales shimmering. With steadfast heart, he raised his sword and struck true. Villagers cheered; the knight returned, honor intact.', type='TextMessage'), TextMessage(source='story_critic', models_usage=RequestUsage(prompt_tokens=177, completion_tokens=594), metadata={}, content='He journeyed back, heart light, knowing valor thrives beyond glory. The tale of Sir Aric spread, inspiring countless souls. **Feedback**: concise, vivid imagery; consider deeper motivation or twist for richer depth.', type='TextMessage'), TextMessage(source='story_writer', models_usage=RequestUsage(prompt_tokens=224, completion_tokens=699), metadata={}, 

In [ ]:
# The team is not stopping immediately but rather current agent is completing its run.

# Aborting A Team

Different from stopping a team, aborting a team will immediately stop the team and raise a CancelledError exception.

In [19]:
from autogen_core import CancellationToken

cancellation_token = CancellationToken()

run2 = asyncio.create_task(
    Console(team.run_stream(task = 'Give a short Story about a lion atmost 40 words',cancellation_token=cancellation_token))
)

await asyncio.sleep(5)
cancellation_token.cancel()

try:
    result = await run2
except :
    print("Task Was Cancelled")

---------- TextMessage (user) ----------
Give a short Story about a lion atmost 40 words
---------- TextMessage (story_writer) ----------
Sir Galen rode through stormy moors, shield gleaming. He faced the dragon that terrorized the valley, striking with valor. The villagers cheered, and his name became legend, inspiring heroes for generations.
---------- TextMessage (story_critic) ----------
In the golden savannah, a young lion named Leo learned to hunt. He chased the pride’s prey, roaring triumph, earning respect. The jungle celebrated his courage, and Leo’s legend lived beyond the dunes.
---------- TextMessage (story_writer) ----------
In the golden savannah, young Leo stalked his first prey, heart pounding. He landed with silent grace, claiming victory. Pride roared his name, and the lion’s courage echoed across the dunes, inspiring future generations.
---------- TextMessage (story_critic) ----------
THE END
---------- TextMessage (story_writer) ----------
THE END


Error processing publish message for story_critic_6b3f96e6-8c31-43bd-8d50-0a6ed3e6353e/6b3f96e6-8c31-43bd-8d50-0a6ed3e6353e
Traceback (most recent call last):
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 604, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/vishalsharma/AI/Autogen/venv/lib/python3.14/site-packages/autogen_cor

Task Was Cancelled
